In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

uploaded = cv2.imread('/content/mari.png') 
gray = cv2.imread('/content/mari.png', cv2.IMREAD_GRAYSCALE)

# Manual 2D convolution
def apply_kernel(image, kernel):
    h, w = image.shape
    kh, kw = kernel.shape
    pad_h, pad_w = kh // 2, kw // 2

    # Pad the image
    padded = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode='constant', constant_values=0)
    result = np.zeros_like(image, dtype=np.float32)

    # Convolve
    for i in range(h):
        for j in range(w):
            region = padded[i:i + kh, j:j + kw]
            result[i, j] = np.sum(region * kernel)
    
    return np.clip(result, 0, 255).astype(np.uint8)

# Kernels
sharpen_kernel = np.array([[0, -1, 0],
                           [-1, 7, -1],
                           [0, -1, 0]])

blur_kernel = np.ones((7, 7), dtype=np.float32) / 49

sobel_x = np.array([[-1, 0, 1],
                    [-2, 0, 2],
                    [-1, 0, 1]])

sobel_y = np.array([[-1, -2, -1],
                    [ 0,  0,  0],
                    [ 1,  2,  1]])

corner_kernel = sobel_x + sobel_y

# Manually implemented convolution
sharpen_manual = apply_kernel(gray, sharpen_kernel)
blur_manual = apply_kernel(gray, blur_kernel)
corner_manual = apply_kernel(gray, corner_kernel)

# OpenCV filter2D

# Blur
blur_kern = np.ones((5, 5), np.float32) / 25
blur_cv = cv2.filter2D(gray, -1, blur_kern)

# Sharpening
sharpen_kernel = np.array([[0, -1, 0],
                           [-1, 5, -1],
                           [0, -1, 0]])
sharpen_cv = cv2.filter2D(gray, -1, sharpen_kernel)

# Corner with sobel
sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
sobel_x = cv2.convertScaleAbs(sobel_x)
sobel_y = cv2.convertScaleAbs(sobel_y)
corner_cv = cv2.addWeighted(sobel_x, 0.5, sobel_y, 0.5, 0)

# Show
titles = ['Sharpen (manual)', 'Sharpen (cv2)',
          'Blur (manual)', 'Blur (cv2)',
          'Corners (manual)', 'Corners (cv2)']
images = [sharpen_manual, sharpen_cv,
          blur_manual, blur_cv,
          corner_manual, corner_cv]

plt.figure(figsize=(15, 8))
for i in range(len(images)):
    plt.subplot(2, 4, i+1)
    plt.imshow(images[i], cmap='gray')
    plt.title(titles[i])
    plt.axis('off')

plt.tight_layout()
plt.show()